# 05 — Reduction ratio and ablation (Tables 1 & 3)

**Tujuan.** Bandingkan nilai paper dengan instansiasi kandidat tanpa training. Table 1 hlm. 91509: r=2,4,8,16,32. Table 3 hlm. 91510: baseline, +MFB, +Attention, +keduanya. Varian runner hanya mengubah field yang dinyatakan; accuracy kandidat tetap `PENDING` sampai run terkunci selesai.

## Mekanisme dan angka yang tersedia

Reduction ratio `r` mengecilkan kanal perantara Coordinate Attention menjadi `max(1,C//r)`. Nilai r lebih besar menurunkan parameter attention kandidat, tetapi tidak membuktikan kinerja lebih baik. MFB menggabungkan fitur empat dilation (3/5/7/9) sebelum fusion; CA memberi gerbang arah tinggi dan lebar. [Diagram dan rincian implementasi](02_architecture.ipynb) adalah referensi struktur, bukan hasil ablasi performa.

| r | Paper Table 1: parameter | Kandidat: parameter struktural | Paper accuracy APTOS | Kandidat accuracy ablasi |
| ---: | ---: | ---: | ---: | --- |
| 2 | 7.9543 M | 5,443,941 | 88.4% | PENDING |
| 4 | 7.5847 M | 5,351,461 | 88.1% | PENDING |
| 8 | 7.3999 M | 5,305,221 | 88.0% | PENDING |
| 16 | 7.2629 M | 5,282,101 | 88.6% | PENDING |
| 32 | 7.2460 M | 5,270,541 | 87.9% | PENDING |

Angka paper adalah **PAPER_REFERENCE**, parameter kandidat berasal dari instansiasi `CADViTCandidate` pada output kode berikut (**SYNTHETIC_TEST / structural analysis**). FLOPs paper dan MAC analitik kandidat memakai konvensi berbeda. Paper Table 3: baseline 83,9%/92,6%, +MFB 86,2%/95,1%, +Attention 87,2%/96,4%, gabungan 88,6%/96,7% (accuracy/specificity). Itu hasil penulis, bukan run ablation kami. `validate_variant_matrix()` memeriksa flag varian, belum melatih dan membandingkan empat kondisi. Hanya model penuh r=16 yang memiliki final test balanced APTOS; ia tidak mengisi baris kinerja ablasi r lain. Status kinerja ablasi tetap **PENDING** sampai protokol, split, checkpoint, dan evaluasi terkunci tersedia.

In [1]:
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "reproduction/cadvit/model.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ["CAD_VIT_ARTIFACT_ROOT"]).expanduser().resolve() if os.environ.get("CAD_VIT_ARTIFACT_ROOT") else None
print("Artifact root:", ARTIFACT_ROOT if ARTIFACT_ROOT else "PENDING (set CAD_VIT_ARTIFACT_ROOT)")

Artifact root: PENDING (set CAD_VIT_ARTIFACT_ROOT)


In [2]:
from reproduction.cadvit.visualization import PAPER_TABLE_1, PAPER_TABLE_3
from reproduction.cadvit.experiments import VARIANTS, core_macs, validate_variant_matrix
from reproduction.cadvit.model import CADViTCandidate
print("Variant flags verified:", validate_variant_matrix())
for reduction, accuracy, paper_mparams, paper_gflops in PAPER_TABLE_1:
    config = VARIANTS[f"aptos_r{reduction}"].model_config()
    candidate_params = sum(p.numel() for p in CADViTCandidate(config).parameters())
    print("PAPER_REFERENCE r", reduction, "accuracy", accuracy, "params(M)", paper_mparams,
          "FLOPs(G)", paper_gflops, "| candidate params", candidate_params,
          "core MAC(G)", round(core_macs(config)["gmac"], 6), "| candidate accuracy PENDING")
print("PAPER_REFERENCE Table 3:", PAPER_TABLE_3)

Variant flags verified: True
PAPER_REFERENCE r 2 accuracy 88.4 params(M) 7.9543 FLOPs(G) 1.982 | candidate params 5443941 core MAC(G) 8.959035 | candidate accuracy PENDING
PAPER_REFERENCE r 4 accuracy 88.1 params(M) 7.5847 FLOPs(G) 1.9753 | candidate params 5351461 core MAC(G) 8.955234 | candidate accuracy PENDING
PAPER_REFERENCE r 8 accuracy 88.0 params(M) 7.3999 FLOPs(G) 1.972 | candidate params 5305221 core MAC(G) 8.953333 | candidate accuracy PENDING
PAPER_REFERENCE r 16 accuracy 88.6 params(M) 7.2629 FLOPs(G) 1.9703 | candidate params 5282101 core MAC(G) 8.952383 | candidate accuracy PENDING
PAPER_REFERENCE r 32 accuracy 87.9 params(M) 7.246 FLOPs(G) 1.9696 | candidate params 5270541 core MAC(G) 8.951908 | candidate accuracy PENDING
PAPER_REFERENCE Table 3: (('baseline', 83.9, 92.6), ('MFB', 86.2, 95.1), ('Attention', 87.2, 96.4), ('MFB+Attention', 88.6, 96.7))


In [3]:
from reproduction.cadvit.visualization import load_controlled_result
if ARTIFACT_ROOT:
    for name in ("aptos_r2", "aptos_r4", "aptos_r8", "aptos_r16", "aptos_r32",
                 "aptos_baseline", "aptos_mfb", "aptos_attention"):
        item = load_controlled_result(ARTIFACT_ROOT, "balanced_v1", name)
        if item["status"] == "AVAILABLE":
            result = item["result"]
            print("CONTROLLED_FINAL_TEST", name, "campaign balanced_v1",
                  "split", result["split"], "checkpoint epoch", result["selected_epoch"],
                  "protocol", result["protocol_sha256"],
                  "manifest", result["manifest_sha256"],
                  "accuracy", result["metrics"]["accuracy"],
                  "specificity_macro", result["metrics"]["specificity_macro"],
                  "result SHA-256", item["result_sha256"])
        else:
            print(name, item["status"], item.get("reason"))
else:
    print("PENDING: controlled balanced_v1 run artifacts unavailable locally")

PENDING: controlled balanced_v1 run artifacts unavailable locally


**Keterbatasan.** Paper tidak menyediakan width/depth lengkap atau konvensi FLOPs. Angka kandidat tidak dipaksa cocok. Full `aptos_r16` dapat dipakai ulang untuk Table 1/3/4 hanya dalam kampanye balanced yang sama; checkpoint V2 BG10 historis mempunyai protokol berbeda.